In [1]:
import warnings
warnings.filterwarnings( 'ignore' )

In [2]:
import pandas as  pd
import numpy as np
import pickle
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from nltk.corpus import stopwords
import nltk
import unicodedata

In [3]:
path = 'D:/GFP/base_apilada_2018_2024.xlsx'
dataoriginal = pd.read_excel(path)

In [147]:
path = 'D:/GFP/diccionario_variables_full.xlsx'
varnames = pd.read_excel(path, engine='openpyxl')

In [145]:
data = dataoriginal

**1. Exploración de la data**

In [149]:
# Filtrar las variables que no deben usarse (usar == 'NO')
vars_to_exclude = varnames[varnames['usar'] == 'no']['variable'].tolist()
data = data.drop(columns=vars_to_exclude)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13153 entries, 0 to 13152
Columns: 189 entries, idmunici to anio
dtypes: float64(182), int64(2), object(5)
memory usage: 19.0+ MB


In [151]:
import pandas as pd

# Asegúrate de que tipo_agregacion esté bien definido (rellenando NaN con 'ninguna')
varnames['tipo_agregacion'] = varnames['tipo_agregacion'].fillna('ninguna')

# Agregar todas las variables dentro de cada bloque según tipo de agregación
for bloque in varnames['bloque'].unique():
    # Filtrar las variables del bloque actual
    bloque_vars = varnames[varnames['bloque'] == bloque]
    
    # Sumar las variables de tipo "sumar" dentro de este bloque
    if 'sumar' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con tipo "sumar"
        vars_a_sumar = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar']['variable']
        
        # Verificar que las variables existan en 'data'
        vars_a_sumar = [var for var in vars_a_sumar if var in data.columns]
        
        # Asegúrate de que las variables son numéricas y tratar casos de "ninguna" o "tesp"
        for var in vars_a_sumar:
            if data[var].isnull().all() or (data[var] == 0).all():  # Si toda la columna tiene NaN o 0
                data[var] = 'ninguna'
        
        # Asegúrate de que las variables son numéricas (excepto "ninguna")
        data[vars_a_sumar] = data[vars_a_sumar].apply(pd.to_numeric, errors='coerce')
        
        if vars_a_sumar:
            # Sumar todas las variables seleccionadas, ignorando NaN y ceros
            data[f'{bloque}_sum'] = data[vars_a_sumar].apply(lambda x: x[x != 0].sum(), axis=1)
            data.drop(columns=vars_a_sumar, inplace=True)  # Eliminar las variables sumadas
    
    # Sumar las operativas dentro del bloque
    if 'sumar_operativas' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con "sumar_operativas"
        vars_operativas = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar_operativas']['variable']
        
        # Verificar que las variables operativas existan en 'data'
        vars_operativas = [var for var in vars_operativas if var in data.columns]
        
        # Asegúrate de que las variables son numéricas y tratar casos de "ninguna" o "tesp"
        for var in vars_operativas:
            if data[var].isnull().all() or (data[var] == 0).all():  # Si toda la columna tiene NaN o 0
                data[var] = 'ninguna'
        
        # Asegúrate de que las variables son numéricas (excepto "ninguna")
        data[vars_operativas] = data[vars_operativas].apply(pd.to_numeric, errors='coerce')
        
        if vars_operativas:
            # Sumar solo las operativas, ignorando NaN y ceros
            data[f'{bloque}_operativas_sum'] = data[vars_operativas].apply(lambda x: x[x != 0].sum(), axis=1)
            data.drop(columns=vars_operativas, inplace=True)  # Eliminar las variables operativas
    
    # Sumar las no operativas dentro del bloque
    if 'sumar_no_operativas' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con "sumar_no_operativas"
        vars_no_operativas = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar_no_operativas']['variable']
        
        # Verificar que las variables no operativas existan en 'data'
        vars_no_operativas = [var for var in vars_no_operativas if var in data.columns]
        
        # Asegúrate de que las variables son numéricas y tratar casos de "ninguna" o "tesp"
        for var in vars_no_operativas:
            if data[var].isnull().all() or (data[var] == 0).all():  # Si toda la columna tiene NaN o 0
                data[var] = 'ninguna'
        
        # Asegúrate de que las variables son numéricas (excepto "ninguna")
        data[vars_no_operativas] = data[vars_no_operativas].apply(pd.to_numeric, errors='coerce')
        
        if vars_no_operativas:
            # Sumar solo las no operativas, ignorando NaN y ceros
            data[f'{bloque}_no_operativas_sum'] = data[vars_no_operativas].apply(lambda x: x[x != 0].sum(), axis=1)
            data.drop(columns=vars_no_operativas, inplace=True)  # Eliminar las variables no operativas
    
  # S# Contar las variables de tipo "tesp" dentro del bloque
    if 'tesp' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con tipo "tesp"
        vars_tesp = bloque_vars[bloque_vars['tipo_agregacion'] == 'tesp']['variable']
    
        # Verificar que las variables tesp existan en 'data'
        vars_tesp = [var for var in vars_tesp if var in data.columns]
    
        # Asegúrate de que las variables son numéricas y tratar casos de "ninguna"
        data[vars_tesp] = data[vars_tesp].apply(pd.to_numeric, errors='coerce')
    
        if vars_tesp:
            # Contar las variables tesp, ignorando NaN, ceros y vacíos
            data[f'tesp_{bloque}_conteo'] = data[vars_tesp].apply(lambda x: ((x != 0) & (x.notna())).sum(), axis=1)
        
            # Eliminar las variables tesp originales después del conteo
            data.drop(columns=vars_tesp, inplace=True)
        
# Ver las primeras filas del dataframe con las sumas realizadas
data.head()

,idmunici,ccdd,ccpp,Departamento,Provincia,Distrito,P19_1_T,P19_2_T,P19_3_T,P19_4_T,...,infraestructura_casa_cultura_sum,casos_atendidos_sum,intervenciones_serenazgo_sum,unidades_moviles_sereno_operativas_sum,unidades_moviles_sereno_no_operativas_sum,equipamiento_sereno_sum,tesp_acciones_realizadas_conteo,tesp_elemento_ambiente_conteo,tesp_instrumento_ambiente_conteo,tesp_acciones_ambiente_conteo
0,10101,1.0,1.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,13.0,58.0,41.0,102.0,...,0.0,281.0,326.0,1.0,0.0,22.0,3,6,0,6
1,10102,1.0,1.0,AMAZONAS,CHACHAPOYAS,ASUNCION,1.0,3.0,1.0,2.0,...,0.0,10.0,3.0,0.0,0.0,4.0,2,1,0,1
2,10103,1.0,1.0,AMAZONAS,CHACHAPOYAS,BALSAS,0.0,3.0,0.0,1.0,...,0.0,7.0,0.0,0.0,0.0,0.0,0,2,1,0
3,10104,1.0,1.0,AMAZONAS,CHACHAPOYAS,CHETO,0.0,2.0,2.0,0.0,...,0.0,10.0,0.0,0.0,0.0,0.0,0,2,2,1
4,10105,1.0,1.0,AMAZONAS,CHACHAPOYAS,CHILIQUIN,0.0,1.0,1.0,1.0,...,0.0,13.0,0.0,0.0,0.0,0.0,0,4,0,0


**5. Filtros pendientes**

In [152]:
# 1. Filtrar Missings: columnas con más del 10% de valores faltantes
threshold = 0.1  # Umbral de valores faltantes permitido
missing_ratio = data.isna().mean()  # Calcula el porcentaje de valores faltantes por columna
data = data.loc[:, missing_ratio <= threshold]  # Mantiene solo las columnas con menos del umbral

In [153]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13153 entries, 0 to 13152
Data columns (total 38 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   idmunici                                   13153 non-null  int64  
 1   ccdd                                       13150 non-null  float64
 2   ccpp                                       13150 non-null  float64
 3   Departamento                               13150 non-null  object 
 4   Provincia                                  13150 non-null  object 
 5   Distrito                                   13150 non-null  object 
 6   P19_1_T                                    12958 non-null  float64
 7   P19_2_T                                    12958 non-null  float64
 8   P19_3_T                                    12958 non-null  float64
 9   P19_4_T                                    12958 non-null  float64
 10  P19_5_T               

In [154]:
# Definir columnas geográficas para la imputación
columnas_geo = ['Departamento', 'Provincia', 'Distrito', 'anio']
variables_num = data.select_dtypes(include=['number']).columns.tolist()
data[variables_num] = data.groupby(columnas_geo)[variables_num].transform(lambda x: x.fillna(x.mean()))

In [155]:
# Paso 2: Imputar los NaN restantes con media global
data[variables_num] = data[variables_num].fillna(data[variables_num].mean())

In [156]:
# 2. Filtro de Variabilidad: Columnas que solo tienen una categoria, esas se eliminan
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
columnas_baja_variabilidad = [col for col in variables_cat if data[col].nunique(dropna=True) == 1]
data = data.drop(columns=columnas_baja_variabilidad)

In [157]:
# Paso 3: Imputación por moda a nivel (departamento, provincia, distrito)
columnas_geo = ['Departamento', 'Provincia', 'Distrito', 'anio']
# Seleccionar variables categóricas a imputar
variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
variables_cat = [col for col in variables_cat if col not in columnas_geo]  # excluir claves geográficas

# Imputar por moda dentro de cada grupo único (departamento, provincia, distrito)
for col in variables_cat:
    try:
        data[col] = data.groupby(columnas_geo)[col].transform(
            lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
        )
    except Exception as e:
        print(f"No se pudo imputar la variable '{col}': {e}")

In [158]:
# Paso 4: Imputación por moda a global (departamento, provincia, distrito)
for col in variables_cat:
    if data[col].isna().sum() > 0:
        try:
            moda_global = data[col].mode().iloc[0]
            data[col] = data[col].fillna(moda_global)
        except Exception as e:
            print(f"No se pudo imputar globalmente la variable '{col}': {e}")

In [159]:
# Paso 5: Filtro de Variabilidad: Columnas que tienen 0.1 por ciento en comparación al total de casos
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
# Umbral de cardinalidad
umbral_cardinalidad = 200
# Identificar columnas de alta cardinalidad (excepto 'distrito')
columnas_alta_cardinalidad = [
    col for col in variables_cat 
    if data[col].nunique(dropna=True) > umbral_cardinalidad and col != 'Distrito'
]
# Eliminar columnas seleccionadas
data = data.drop(columns=columnas_alta_cardinalidad, errors='ignore')

In [160]:
# Convertir montos a log
# Crear versiones en log
data["log_C96"] = np.log1p(data["C96"])  # log(1+x)
data["log_C97"] = np.log1p(data["C97"])

# (Opcional) Eliminar las originales
data = data.drop(columns=["C96", "C97"])

In [161]:
#2. Merge con la data

In [162]:
path = 'D:/GFP/1_data_contrata.xlsx'
data2 = pd.read_excel(path, engine='openpyxl')

In [163]:
data_filtrada = data2

In [139]:
data_filtrada.columns.tolist()

['marca_reconstruccion',
 'marca_reactivacion',
 'n_informes_monitores',
 'n_denuncias',
 'n_informes_control',
 'n_comentarios_ciudadanos',
 'n_obras_relacionadas',
 'monto_aprobado_soles',
 'plazo_ejecucion_dias',
 'porcentaje_terreno_entregado',
 'avance_fisico_real',
 'porcentaje_ejecucion_financiera',
 'estado_actualizacion_avance',
 'existe_paralizacion',
 'n_modificaciones',
 'n_controversias',
 'n_adicionales_obra',
 'n_adicionales_supervision',
 'n_deductivos_obra',
 'n_deductivos_supervision',
 'anio_inicio_obra',
 'naturaleza_obra_Construcción/Creación',
 'naturaleza_obra_Descolmatación',
 'naturaleza_obra_Encauzamiento',
 'naturaleza_obra_Habilitación',
 'naturaleza_obra_Instalación',
 'naturaleza_obra_Limpieza',
 'naturaleza_obra_Mejoramiento',
 'naturaleza_obra_Reconstrucción',
 'naturaleza_obra_Recuperación',
 'naturaleza_obra_Rehabilitación',
 'naturaleza_obra_Remodelación',
 'naturaleza_obra_Renovación',
 'naturaleza_obra_Reparación',
 'naturaleza_obra_Reposición',
 'R

In [183]:
# renombrar
rename_map = {
    'provincia_norm': 'Provincia',
    'departamento':   'Departamento',
    'distrito':       'Distrito',
    'anio_inicio_obra':'anio',   # <- solo esta cambia a Year
    # 'brecha_existente' se queda igual
}
data_filtrada = data_filtrada.rename(columns=rename_map)

print(data_filtrada.head())

   marca_reconstruccion  marca_reactivacion  n_informes_monitores  \
0                     0                   0                     0   
1                     0                   0                     0   
2                     0                   0                     0   
3                     0                   0                     0   
4                     0                   0                     0   

   n_denuncias  n_informes_control  n_comentarios_ciudadanos  \
0            0                   1                         0   
1            0                   1                         0   
2            0                   1                         0   
3            0                   1                         0   
4            0                   1                         0   

   n_obras_relacionadas  monto_aprobado_soles  plazo_ejecucion_dias  \
0                     2          4.726207e+05                    90   
1                     3          1.320000e+05             

In [185]:
data_filtrada.columns.tolist()

['marca_reconstruccion',
 'marca_reactivacion',
 'n_informes_monitores',
 'n_denuncias',
 'n_informes_control',
 'n_comentarios_ciudadanos',
 'n_obras_relacionadas',
 'monto_aprobado_soles',
 'plazo_ejecucion_dias',
 'porcentaje_terreno_entregado',
 'avance_fisico_real',
 'porcentaje_ejecucion_financiera',
 'estado_actualizacion_avance',
 'existe_paralizacion',
 'n_modificaciones',
 'n_controversias',
 'n_adicionales_obra',
 'n_adicionales_supervision',
 'n_deductivos_obra',
 'n_deductivos_supervision',
 'anio',
 'naturaleza_obra_Construcción/Creación',
 'naturaleza_obra_Descolmatación',
 'naturaleza_obra_Encauzamiento',
 'naturaleza_obra_Habilitación',
 'naturaleza_obra_Instalación',
 'naturaleza_obra_Limpieza',
 'naturaleza_obra_Mejoramiento',
 'naturaleza_obra_Reconstrucción',
 'naturaleza_obra_Recuperación',
 'naturaleza_obra_Rehabilitación',
 'naturaleza_obra_Remodelación',
 'naturaleza_obra_Renovación',
 'naturaleza_obra_Reparación',
 'naturaleza_obra_Reposición',
 'Region_costa 

In [187]:
# --- Normalizar textos ---
def normalize_text(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return None if s in ['NAN','NULL','NONE',''] else s

# Normalizar claves en ambas bases
for col in ['Provincia', 'Distrito', 'Departamento']:
    data[col] = data[col].map(normalize_text)
    data_filtrada[col] = data_filtrada[col].map(normalize_text)

# Asegurar que el año sea string sin decimales
data["anio"] = data["anio"].astype(int).astype(str)
data_filtrada["anio"] = data_filtrada["anio"].astype(int).astype(str)

# Crear variable única en cada dataset
data["dep_prov_dist"] = (
    data["Departamento"] + "_" +
    data["Provincia"] + "_" +
    data["Distrito"] + "_" +
    data["anio"]
)

data_filtrada["dep_prov_dist"] = (
    data_filtrada["Departamento"] + "_" +
    data_filtrada["Provincia"] + "_" +
    data_filtrada["Distrito"] + "_" +
    data_filtrada["anio"]
)

In [191]:
# Diccionario de corrección de errores comunes
correcciones = {
    # ANCASH mal con Andahuaylas (que es de Apurímac)
    "ANCASH_ANDAHUAYLAS_MATO": "APURIMAC_ANDAHUAYLAS_MATO",
    "ANCASH_ANDAHUAYLAS_PAMPAROMAS": "ANCASH_HUAYLAS_PAMPAROMAS",
    "ANCASH_ANDAHUAYLAS_SANTA CRUZ": "ANCASH_SANTA_SANTA CRUZ",
    "ANCASH_ANDAHUAYLAS_PUEBLO LIBRE": "ANCASH_HUAYLAS_PUEBLO LIBRE",

    # Errores de escritura con Ñ / tildes
    "NEPE A": "NEPENA",
    "BA OS": "BANOS",
    "CASTA EDA": "CASTANEDA",
    "ENCA ADA": "ENCANADA",
    "QUI OTA": "QUINOTA",
    "SA O": "SANO",
    "PU OS": "PUNOS",
    "NU OA": "NUNOA",
    "MA AZO": "MANAZO",
    "CU UMBUQUI": "CUNUMBUQUI",
    "PARI AS": "PARINAS",

    # Casos de departamento/provincia mal asignados
    "HUANCAVELICA_CHINCHA": "ICA_CHINCHA",
    "AREQUIPA_AREQUIPA_PUCYURA": "CUSCO_ANTA_PUCYURA",

    # Casos de concatenación doble
    "LIMA_LIMA_PUEBLO LIBRE MAGDALENA VIEJA": "LIMA_LIMA_PUEBLO LIBRE"
}


In [193]:
import re

def corregir_clave(clave):
    for error, correccion in correcciones.items():
        if error in clave:
            return re.sub(error, correccion, clave)
    return clave

# Aplicar correcciones sobre dep_prov_dist
data_filtrada["dep_prov_dist"] = data_filtrada["dep_prov_dist"].apply(corregir_clave)


In [235]:
# --- Merge directo ---
merged = data_filtrada.merge(
    data,
    on="dep_prov_dist",   # clave única
    how="left",           # todas las obras, aunque no tengan municipio
    suffixes=("_obra", "_muni")
)


In [237]:
# Chequear resultados
print("Total obras:", len(data_filtrada))
print("Obras con match:", merged["Departamento_muni"].notna().sum())
print("Obras sin match:", merged["Departamento_muni"].isna().sum())

Total obras: 20597
Obras con match: 20158
Obras sin match: 439


In [ ]:
## Tratamiento de los sin match

In [239]:
# Filtrar proyectos sin match
obras_sin_match = merged[merged["Departamento_muni"].isna()]

print("Número de obras sin match:", len(obras_sin_match))

Número de obras sin match: 439


In [107]:
#obras_sin_match.to_excel("obras_sin_match.xlsx", index=False)

In [91]:
#merged.to_excel("merged_obras_municipios.xlsx", index=False)

In [241]:
from thefuzz import process
import pandas as pd

# 1. Filtrar obras sin match
obras_sin_match = merged[merged["Departamento_muni"].isna()].copy()

# 2. Claves posibles de municipios
choices = data["dep_prov_dist"].astype(str).unique().tolist()

# 3. Aplicar fuzzy match solo a las obras sin match
matches = {}
for key in obras_sin_match["dep_prov_dist"].astype(str).unique():
    match, score = process.extractOne(key, choices)
    matches[key] = (match, score)

# 4. Pasar a DataFrame
matches_df = pd.DataFrame.from_dict(
    matches, orient="index", columns=["best_match", "score"]
).reset_index().rename(columns={"index": "dep_prov_dist"})


In [242]:
matches_filtrados = matches_df[matches_df["score"] > 95]
matches_filtrados

,dep_prov_dist,best_match,score
1,LIMA_HUAURA_CAHUACHO_2022,LIMA_HUAURA_HUACHO_2022,96
5,APURIMAC_CHINCHEROS_ANCO HUALLO_2018,APURIMAC_CHINCHEROS_ANCO-HUALLO_2018,100
6,JUNIN_CHUPACA_SAN JUAN DE YSCOS_2018,JUNIN_CHUPACA_SAN JUAN DE ISCOS_2018,97
8,ICA_ICA_LA TINGUI A_2018,ICA_ICA_LA TINGUINA_2018,96
10,ICA_NAZCA_EL INGENIO_2018,ICA_NASCA_EL INGENIO_2018,96
...,...,...,...
99,UCAYALI_PADRE ABAD_HUIPOCA_2022,UCAYALI_PADRE ABAD_HUIPOCA_2023,97
102,ANCASH_BOLOGNESI_ANTONIO RAIMONDI_2022,ANCASH_BOLOGNESI_ANTONIO RAYMONDI_2022,97
134,LA LIBERTAD_PATAZ_TURPAY_2023,LA LIBERTAD_PATAZ_URPAY_2023,98
139,JUNIN_YAULI_HUAY HUAY_2023,JUNIN_YAULI_HUAY-HUAY_2023,100


In [245]:
obras_sin_match = obras_sin_match.merge(
    matches_filtrados,
    on="dep_prov_dist",
    how="left"
)

In [255]:
obras_sin_match["Departamento_muni"] = obras_sin_match["best_match"]

In [257]:
merged_final = pd.concat([
    merged[merged["Departamento_muni"].notna()],  # los que ya tenían match
    obras_sin_match                               # los corregidos por fuzzy
])

In [259]:
print("Total obras:", len(merged_final))
print("Obras con match:", merged_final["Departamento_muni"].notna().sum())
print("Obras sin match:", merged_final["Departamento_muni"].isna().sum())


Total obras: 20597
Obras con match: 20366
Obras sin match: 231


In [213]:
#matches_filtrados.to_excel("matches_score_mayor_95.xlsx", index=False)

In [ ]:
###. PASO OPCIONNAL!!!

In [95]:
# Variables que quieres excluir del filtro, pero que sí deben estar en el dataframe final
excluir = [
    "Provincia_obra",
    "Distrito_obra",
    "Departamento_obra",
    "brecha_dias",
    "dep_prov_dist",
    "idmunici",
    "ccdd",
    "ccpp",
    "Departamento_muni",
    "Provincia_muni",
    "Distrito_muni"
]

# Paso 1: Matriz de correlación (solo numéricas)
corr_matrix = merged.corr(numeric_only=True)

# Paso 2: Correlación de cada variable con la dependiente
cor_with_y = corr_matrix['brecha_existente'].drop('brecha_existente', errors='ignore')

# Paso 3: Variables independientes excluyendo las de 'excluir'
indep_vars = [v for v in cor_with_y.index if v not in excluir]
corr_indep = corr_matrix.loc[indep_vars, indep_vars]

# Paso 4: Eliminar variables muy correlacionadas (>= 0.85)
vars_to_remove = set()
threshold = 0.85
for i in range(len(indep_vars)):
    for j in range(i + 1, len(indep_vars)):
        var1, var2 = indep_vars[i], indep_vars[j]
        r = abs(corr_indep.loc[var1, var2])
        if r >= threshold:
            if abs(cor_with_y[var1]) >= abs(cor_with_y[var2]):
                vars_to_remove.add(var2)
            else:
                vars_to_remove.add(var1)

# Paso 5: Variables seleccionadas después del filtro
vars_selected = [v for v in indep_vars if v not in vars_to_remove]

# Paso 6: DataFrame final con seleccionadas + dependiente + excluidas
cols_finales = vars_selected + ['brecha_existente'] + [c for c in excluir if c in data.columns]
data_preseleccionada = merged[cols_finales].copy()


In [261]:
#data.to_csv("full_data.csv", index=False, encoding="latin1")

In [97]:
data_preseleccionada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20597 entries, 0 to 20596
Columns: 227 entries, marca_reconstruccion to ccpp
dtypes: float64(37), int64(189), object(1)
memory usage: 35.7+ MB


In [99]:
import pandas as pd
# Lista para guardar los detalles de exclusión
exclusion_log = []

# Repetimos la lógica del filtro, pero esta vez guardando la info de cada exclusión
for i in range(len(indep_vars)):
    for j in range(i + 1, len(indep_vars)):
        var1 = indep_vars[i]
        var2 = indep_vars[j]
        r = abs(corr_indep.loc[var1, var2])
        if r >= threshold:
            cor1 = abs(cor_with_y[var1])
            cor2 = abs(cor_with_y[var2])
            excluida = var2 if cor1 >= cor2 else var1
            conservada = var1 if excluida == var2 else var2
            exclusion_log.append({
                "var1": var1,
                "var2": var2,
                "correlacion_entre_ellas": r,
                "cor_var1_con_brecha": cor1,
                "cor_var2_con_brecha": cor2,
                "variable_conservada": conservada,
                "variable_excluida": excluida
            })

# Convertir a DataFrame
exclusion_df = pd.DataFrame(exclusion_log)
# Exportar a Excel
exclusion_df.to_excel("detalle_variables_excluidas.xlsx", index=False)

In [103]:
data_preseleccionada.to_excel("1_data_contrata_renamu.xlsx", index=False, engine="openpyxl")